<a href="https://colab.research.google.com/github/Shivam-vachhani/PyTorch-Projects/blob/main/CNN_on_Fashion_MNIST_Dataset_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [59]:
df = pd.read_csv('fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [61]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [62]:
X_train = X_train/255.0
X_test = X_test/255.0

In [63]:
class MyCustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [64]:
train_dataset = MyCustomDataset(X_train,y_train)
test_dataset = MyCustomDataset(X_test,y_test)

In [65]:
# train_dataset[0]

In [76]:
# Making CNN by seprate task first feature extraction by CNN and second is classification by ANN

class MyNN(nn.Module):

  def __init__(self,num_channels):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(num_channels,32,kernel_size=3,padding="same"),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(32,64,kernel_size=3,padding="same"),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2),
    )

    self.classification = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(64,10),
    )


  def forward(self,x):
    x = self.features(x)
    x = self.classification(x)
    return x


In [77]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [78]:
epoch = 50
learning_rate = 0.01

In [82]:
# Initialize Model
model = MyNN(1)

# loss function
criterion = nn.CrossEntropyLoss()

# Optimizer function
optimizer = optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)


In [81]:
for i in range(epoch):
  total_loss = 0
  for batch_features,batch_labels in train_loader:

    #forward pass
    output = model(batch_features)

    #loss calulate
    loss = criterion(output,batch_labels)

    #backward pass
    optimizer.zero_grad()
    loss.backward()

    #update parameters
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch {i+1} Loss: {total_loss/len(train_loader)}")

Epoch 1 Loss: 0.6859486836095651
Epoch 2 Loss: 0.4179101057102283
Epoch 3 Loss: 0.3577495632171631
Epoch 4 Loss: 0.32268374066551525
Epoch 5 Loss: 0.2976595913444956
Epoch 6 Loss: 0.2808967965058982
Epoch 7 Loss: 0.26395801706115407
Epoch 8 Loss: 0.25037478088835874
Epoch 9 Loss: 0.24070743141199152
Epoch 10 Loss: 0.23176679983114204
Epoch 11 Loss: 0.21975984428202114
Epoch 12 Loss: 0.21212446683645247
Epoch 13 Loss: 0.20405072751858583
Epoch 14 Loss: 0.19468171478870014
Epoch 15 Loss: 0.1878837087924282
Epoch 16 Loss: 0.18074224514638385
Epoch 17 Loss: 0.17393048724966745
Epoch 18 Loss: 0.16762723039090632
Epoch 19 Loss: 0.1597817208422348
Epoch 20 Loss: 0.15330860367231072
Epoch 21 Loss: 0.1512465777285397
Epoch 22 Loss: 0.1418470183731988
Epoch 23 Loss: 0.14091189154361686
Epoch 24 Loss: 0.13260705879268547
Epoch 25 Loss: 0.12807839970166485
Epoch 26 Loss: 0.12594065608503296
Epoch 27 Loss: 0.11922834563410531
Epoch 28 Loss: 0.11459629717748612
Epoch 29 Loss: 0.11254854566634943
Epo

In [85]:
# Calculating accuracy

model.eval()

total = 0
correct = 0
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    output = model(batch_features)
    _,predicted = torch.max(output,1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")



Accuracy: 0.09575
